# Text Mining

Topic analysis and sentiment comparison across 4 music subreddits.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.corpus import stopwords

import spacy
from gensim import corpora
from gensim.models import LdaModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download('punkt_tab', quiet=True)

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
sns.set_theme(style="whitegrid")
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/all_subreddits_sample.csv")
print(df.shape)
df.head()

(40000, 10)


,comment_id,author,body,subreddit,score,created_utc,parent_id,link_id,controversiality,gilded
0,ctnf6g4,WakaIsMyWaifu,Lil B's stuff is genuinely enjoyable though,hiphopheads,-1,1438387200,t1_ctneflp,t3_3fcixz,1,0
1,ctnf6oq,n8-dohg,Or you don't got any reasoning behind that bul...,hiphopheads,6,1438387213,t1_ctneet0,t3_3fcixz,0,0
2,ctnf6xt,Big_E33,so many great potential gifs from that vid,hiphopheads,1,1438387225,t1_ctndpm1,t3_3fcixz,0,0
3,ctnf6y4,Keeks_marone,youve got some keen eyes detective,hiphopheads,3,1438387226,t1_ctnf2as,t3_3fc5yn,0,0
4,ctnf78m,MrBokbagok,id let her fart in my dinner TONIGHT,hiphopheads,1,1438387242,t1_ctmzh29,t3_3fagto,0,0


## 1. Preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))

# dodatkowe słowa które nie wnoszą nic do analizy muzycznej
extra_stopwords = {
    "like", "just", "get", "got", "one", "know", "think", "really",
    "would", "also", "even", "much", "still", "way", "good", "make",
    "people", "time", "music", "song", "songs", "album", "artist",
    "https", "http", "www", "com", "reddit", "amp"
}
stop_words.update(extra_stopwords)


def preprocess(text):
    if not isinstance(text, str):
        return []

    doc = nlp(text.lower())
    tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and token.lemma_ not in stop_words
        and len(token.lemma_) > 2
    ]
    return tokens


# lemmatyzacja może chwilę potrwać
df["tokens"] = df["body"].apply(preprocess)

print("Przykład tokenów:")
print(df["tokens"].iloc[0])

## 2. Najczęstsze słowa i bigramy

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (sub, group) in enumerate(df.groupby("subreddit")):
    all_tokens = [token for tokens in group["tokens"] for token in tokens]
    top_words = Counter(all_tokens).most_common(15)

    words, counts = zip(*top_words)
    axes[i].barh(list(words)[::-1], list(counts)[::-1], color="steelblue")
    axes[i].set_title(f"r/{sub} — top 15 słów")
    axes[i].set_xlabel("Liczba wystąpień")

plt.tight_layout()
plt.savefig("../outputs/figures/top_words.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (sub, group) in enumerate(df.groupby("subreddit")):
    all_tokens = [token for tokens in group["tokens"] for token in tokens]
    bigrams = list(ngrams(all_tokens, 2))
    top_bigrams = Counter(bigrams).most_common(15)

    labels = [f"{a} {b}" for (a, b), _ in top_bigrams]
    counts = [c for _, c in top_bigrams]

    axes[i].barh(labels[::-1], counts[::-1], color="steelblue")
    axes[i].set_title(f"r/{sub} — top 15 bigramów")
    axes[i].set_xlabel("Liczba wystąpień")

plt.tight_layout()
plt.savefig("../outputs/figures/top_bigrams.png", dpi=150)
plt.show()

## 3. Word Clouds

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (sub, group) in enumerate(df.groupby("subreddit")):
    all_tokens = [token for tokens in group["tokens"] for token in tokens]
    text = " ".join(all_tokens)

    wc = WordCloud(width=800, height=400, background_color="white", max_words=100)
    wc.generate(text)

    axes[i].imshow(wc, interpolation="bilinear")
    axes[i].axis("off")
    axes[i].set_title(f"r/{sub}", fontsize=14)

plt.tight_layout()
plt.savefig("../outputs/figures/wordclouds.png", dpi=150)
plt.show()

## 4. Modelowanie tematów (LDA)

In [ ]:
NUM_TOPICS = 5
NUM_WORDS  = 8

lda_results = {}

for sub, group in df.groupby("subreddit"):
    tokens_list = group["tokens"].tolist()
    tokens_list = [t for t in tokens_list if len(t) > 0]

    dictionary = corpora.Dictionary(tokens_list)
    # pomijamy słowa które pojawiają się rzadziej niż 5 razy lub w ponad 50% dokumentów
    dictionary.filter_extremes(no_below=5, no_above=0.5)

    corpus = [dictionary.doc2bow(tokens) for tokens in tokens_list]

    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=NUM_TOPICS,
        random_state=42,
        passes=10,
    )

    lda_results[sub] = {"model": lda, "corpus": corpus, "dictionary": dictionary}

    print(f"\nr/{sub}:")
    for topic_id, words in lda.print_topics(num_topics=NUM_TOPICS, num_words=NUM_WORDS):
        print(f"  Temat {topic_id + 1}: {words}")

In [ ]:
# wizualizacja wag słów dla każdego tematu
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (sub, result) in enumerate(lda_results.items()):
    lda = result["model"]
    ax = axes[i]

    for topic_id in range(NUM_TOPICS):
        top = lda.show_topic(topic_id, topn=5)
        words  = [w for w, _ in top]
        weights = [p for _, p in top]
        y_pos = [topic_id * 6 + j for j in range(len(words))]
        bars = ax.barh(y_pos, weights, color=f"C{topic_id}")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=8)

    ax.set_title(f"r/{sub} — tematy LDA")
    ax.set_xlabel("Waga")

plt.tight_layout()
plt.savefig("../outputs/figures/lda_topics.png", dpi=150)
plt.show()

## 5. Analiza sentymentu (VADER)

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# VADER działa na surowym tekście, nie na tokenach
df["sentiment"] = df["body"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"] if isinstance(x, str) else 0
)

# compound to wynik od -1 (bardzo negatywny) do +1 (bardzo pozytywny)
sentiment_stats = df.groupby("subreddit")["sentiment"].agg(["mean", "median"]).round(3)
print(sentiment_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# rozkład sentymentu per subreddit
for sub, group in df.groupby("subreddit"):
    sns.kdeplot(group["sentiment"], label=sub, ax=axes[0])

axes[0].set_title("Rozkład sentymentu")
axes[0].set_xlabel("VADER compound score")
axes[0].set_ylabel("Gęstość")
axes[0].legend()

# średni sentyment per subreddit
means = df.groupby("subreddit")["sentiment"].mean().sort_values()
axes[1].barh(means.index, means.values, color="steelblue")
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=1)
axes[1].set_title("Średni sentyment per subreddit")
axes[1].set_xlabel("VADER compound score")

plt.tight_layout()
plt.savefig("../outputs/figures/sentiment.png", dpi=150)
plt.show()

In [ ]:
# udział komentarzy pozytywnych, neutralnych i negatywnych
def classify_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment"].apply(classify_sentiment)

sentiment_dist = (
    df.groupby(["subreddit", "sentiment_label"])
    .size()
    .unstack()
    .div(df.groupby("subreddit").size(), axis=0)
    .mul(100)
    .round(1)
)

sentiment_dist.plot(kind="bar", figsize=(10, 5), color=["#d9534f", "#aaaaaa", "#5cb85c"])
plt.title("Rozkład sentymentu (%)")
plt.xlabel("Subreddit")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.legend(title="Sentyment")
plt.tight_layout()
plt.savefig("../outputs/figures/sentiment_dist.png", dpi=150)
plt.show()

print(sentiment_dist)

## Wnioski

*(uzupełnij po uruchomieniu)*

- ...
- ...
- ...